# GRM vs PC-Relate kinship, coloured by round-2 PC distance between i and j
Individual pairs: notebook only, nothing saved.

In [ ]:
RUN_ID = "sim"
FORCE = False

In [ ]:
import json, os, subprocess, sys, tempfile
REPO = os.getcwd()
while not os.path.exists(os.path.join(REPO, "pyproject.toml")):
    REPO = os.path.dirname(REPO)
sys.path[:0] = [REPO, os.path.join(REPO, "src")]
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from aoucov import load_run
from aoucov.manifest import needs_run, require_ready, single_target, target_items, stage_status
from aoucov.provenance import collect_meta, record, write_meta
from aoucov.report import Report
from aoucov.store import get_store

cfg = load_run(RUN_ID)
THREADS = min(cfg.params.get("threads", 4), os.cpu_count() or 1)
for v in ("OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"):   # bigsnpr parallelises itself; one BLAS thread (not OMP: it caps bigsnpr too)
    os.environ[v] = "1"
os.environ.pop("OMP_NUM_THREADS", None)
if cfg.params.get("scratch"):   # mktemp in %%bash, tempfile, and R use TMPDIR
    import tempfile
    tempfile.tempdir = os.environ["TMPDIR"] = os.path.expanduser(cfg.params["scratch"])
    os.makedirs(tempfile.tempdir, exist_ok=True)

def setenv(**kw):
    """Variables for the %%bash / %%script cells below."""
    os.environ.update({k: str(v) for k, v in kw.items()})

def mark_done(stage, uri, **extra):
    """Provenance sidecar for an output a shell or R cell just wrote."""
    write_meta(get_store(uri), uri, collect_meta(cfg, stage, uri, extra=extra))

In [ ]:
OUT = single_target(cfg, "pcrelate")
ids = pd.read_csv(OUT.replace(".bin", ".id"), header=None, dtype=str)[0]
n = len(ids)
kin = np.fromfile(OUT, dtype="float32")
grm = np.concatenate([np.fromfile(u, dtype="float32") for u, _ in target_items(cfg, "grm_shards")])
assert len(kin) == len(grm) == n * (n + 1) // 2

K2 = cfg.param("r2_k")
r2 = pd.read_csv(single_target(cfg, "round2_pca"), sep="\t", dtype={"id": str})
r2 = r2[r2["set"] == "cohort"].drop_duplicates("id").set_index("id")
pcs2 = [p for p in r2 if p.startswith("PC")][:K2]
X = r2.reindex(ids)[pcs2].to_numpy()
print(n, "people,", int(np.isnan(X).any(axis=1).sum()), "without round-2 scores")

In [ ]:
rng = np.random.default_rng(1)
i = rng.integers(1, n, 300_000); j = (rng.random(len(i)) * i).astype(int)   # random pairs, i > j
rel = np.flatnonzero((grm / 2 > 0.04) | (kin > 0.04))                          # plus every pair that looks related
ri = ((np.sqrt(8 * rel.astype(float) + 1) - 1) // 2).astype(np.int64); rj = rel - ri * (ri + 1) // 2
keep = ri != rj
i, j = np.r_[i, ri[keep]], np.r_[j, rj[keep]]
t = i * (i + 1) // 2 + j
d = np.linalg.norm(X[i] - X[j], axis=1)
ok = ~np.isnan(d)
g, k, d = grm[t][ok] / 2, kin[t][ok], d[ok]
o = np.argsort(d)   # far pairs drawn last

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, lim in zip(axes, [(-0.06, 0.06), (-0.06, 0.55)]):
    sc = ax.scatter(g[o], k[o], c=d[o], s=2, cmap="viridis", alpha=0.6, rasterized=True)
    ax.plot(lim, lim, "r:", lw=0.8)
    ax.set(xlim=lim, ylim=lim, xlabel="GRM / 2", ylabel="PC-Relate kinship")
axes[0].set_title(f"{RUN_ID}: bulk"); axes[1].set_title("with relatives")
fig.colorbar(sc, ax=axes, label=f"distance in round-2 PC space (PC1-{K2})")
plt.show()